This notebook is for reworking the Higgins' group LCMS target data processing script into Python.

The goal is to make a python script that will be user friendly, clear, and modular. The code will be heavily commented to attempt to document what each line is actually doing and why.

In [ ]:
import os
import pandas as pd
import numpy as np


### SECTION 1 ###

#GENERAL NOTES FOR USERS GO HERE

#Acronyms and Abbreviations
#   DB: Double Blank
#   MB: Method Blank
#   LB: Laboratory Blank
#   LCS: Laboratory Control Sample
#   LLLCS: Low Level Laboratory Control Sample
#   LCSD: ??
#   CCV: Continuing Calibration Verification
#   ISC: Instrument Sensitivity Check
#   IS: Internal Standard
#   InjS: Injection Standard
#   aQC: Analytical Quality Control
#   mQC: Method Quality Control
#   cal: calibration standards, i.e. your calibration curve
#   aQC: CCV, ISC, LB (analytical QC samples)
#       Continuing Calibration Verifications
#       Instrument Sensitivity Check
#       Laboratory Blank
#   mQC: LCS, LLLCS, LCSD, MB (method QC samples)
#       Laboratory Control Sample
#       Low-Level Laboratory Control Sample
#       LCSD?
#       Method Blank


### SECTION 2 ###

##Inputs and Options:

#here you will type the file name (INCLUDING the filetype extension) exactly
file_name = '20260424_ES_USCneg.txt'

#here you will type the path name. If you don't know how to do this: In Windows you can "Copy as Path." On Mac you right click with Option held or you right click then hold option before clicking "copy."
##Future goal: make it so you put the script in a top level folder and it cycles through all folders instead of requiring a hard input of the path name
path_name = r"C:\Users\richa\OneDrive - Colorado School of Mines\Documents\LCMS_data"

# Subtract MB (Method Blank) values from samples to correct for background contamination
subMB = False # False = Off, True = On

# In case of small volume injection direct injection run
direct_inject = True #True for a direct injection, false if SPE method. By default, the previous script assumed all SVI runs to be SPE and all LVI runs to be Direct Inject.

### SECTION 3 ###

#This section will read the file into a Pandas data frame

#Here the script determines if the input is a text file or excel file (SCIEX outputs text, the orbitrap outputs Excel)
name, ext = os.path.splitext(file_name)
if ext not in (".txt", ".xlsx"):
    raise ValueError(f"Error: unsupported filetype {ext}, bust be .txt or .xslx")


#this line will import the output file into pandas as a data frame
if ext == ".txt":
    df = pd.read_csv(os.path.join(path_name, file_name), sep="\t", dtype=str, keep_default_na=False)
elif ext == ".xlsx":
    df = pd.read_excel(os.path.join(path_name, file_name), dtype=str, keep_default_na=False)

# Subtract MB (Method Blank) values from samples to correct for background contamination
subMB = False # False = Off, True = On

# In case of small volume injection direct injection run
direct_inject = True #True for a direct injection, false if SPE method. By default, the previous script assumed all SVI runs to be SPE and all LVI runs to be Direct Inject.

## Checking column names within the data frame
# this is analogous to section 3 of the MatLab script, but pandas doesn't need to do section 3.
# this is just a check for readability

required_cols = ["Sample Name", "Sample Index", "Injection Volume", "Component Name", "Component Group Name", "Component Type", "Retention Time", "Precursor Mass", "Mass Error Confidence", "Mass Error (ppm)", "IS Name", "Area", "IS Area", "Actual Concentration", "Calculated Concentration", "Sample Type", "Used", "Polarity"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in data file")



### SECTION 4 ###

## Determine Analysis Method and QC's (Direct injection vs SPE)
#IMPORTANT, previous code differentiates method by injection volume. Make sure to specify whether or not your method is direct inject or SPE above if you ran a SVI Direct Inject!

method_map = {
    1000: "1000uL",
    100: "100uL",
}

inj_vol = int(df["Injection Volume"].iloc[0])


if inj_vol not in method_map:
    raise ValueError (f"Error: unknown analysis method (injection volume {inj_vol} uL not recognized)")

polarity = df["Polarity"].iloc[1]

meth = method_map[inj_vol]

if meth == "100uL" and direct_inject:
    meth = "100uL_DI"

if meth == "100uL" and polarity == "Negative":
    CCVactconc = 250 #CCV/QC concentration [ng/L]
    ISCactconc = 25  #ISC concentration [ng/L]
    EPAactconc = 250 #EPA Bullseye [ng/L] SHOULD be 250

if meth == "100uL" and polarity == "Positive":
    CCVactconc = 200 #CCV/QC concentration [ng/L]
    ISCactconc = 10  #ISC concentration [ng/L]

if meth == "1mL":
    CCVactconc = 200    #CCV/QC concentration [ng/L]
    ISCactconc = 33.33  #ISC concentration [ng/L]
    EPAactconc = 333.33 #EPA Bullseye [ng/L] 


### SECTION 5 ###

#section 5 from matlab script not needed in python. That section was for handling an issue with Windows MatLab that is bypassed using Pandas.

## Identify Number of Compounds and Samples
#builds a new data frame from just the first sample to extract compound names, precursor masses, component group name, and the Internal Standard assigned to that compound
first_sample = df[df["Sample Name"] == df["Sample Name"].iloc[0]]

targets = first_sample[
    (first_sample["Component Type"].isin(["Quantifiers", "Qualifiers"])) &
    (first_sample["Component Group Name"] != "Inj Stds")
]

comnamelist = targets["Component Name"].tolist()
commass     = targets["Precursor Mass"].tolist()
comgroup    = targets["Component Group Name"].tolist()
comISname   = targets["IS Name"].tolist()

if meth == "100uL":
    inj_stds = first_sample[first_sample["Component Group Name"] == "Inj Stds"]
    InjSnamelist = inj_stds["Component Name"].tolist()
else:
    InjSnamelist = []

numcom = len(comnamelist)
numIS = len(first_sample[
    first_sample["Component Type"].isin(["Internal Standard", "Internal Stardards"]) &
    (first_sample["Component Group Name"] != "Inj Stds")
])


In [ ]:
## Testing What I've Got So Far ##

print(f"Method: {meth}")
print(f"Polarity: {polarity}")
print(f"Number of target compounds {numcom}")
print(f"Number of internal standards {numIS}")
print(f"Compounds: {comnamelist[:5]}")
print(f"Injection Standards: {InjSnamelist}")
print(f"CCV actual concentration: {CCVactconc}")

In [ ]:
### SECTION 6 ###

#Data extraction step
#Categories:
#   DB: double blanks
#   cal: calibration standards
#   aQC: CCV, ISC, LB (analytical QC samples)
#       Continuing Calibration Verifications
#       Instrument Sensitivity Check
#       Laboratory Blank
#   mQC: LCS, LLLCS, LCSD, MB (method QC samples)
#       Laboratory Control Sample
#       Low-Level Laboratory Control Sample
#       LCSD?
#       Method Blank
#   Extract actual calibration concentration, actual IS conc, IS peak area,
#   measured concentration. If value is not a number, it is recorded as NaN.
#   Common non-numerical values include 'N/A', '<0', etc

##Extract Value function##

#def extractvalue(val):
    # if val is None:
    #     return np.nan
    # if isinstance(val, float) and np.isnan(val):
    #     return np.nan
    # if isinstance(val, str):
    #     stripped = val.strip()
    #     if stripped in ("N/A", "<0", ""):
    #         return np.nan
    #     try:
    #         return float(stripped)
    #     except ValueError:
    #         return np.nan
    # return float(val)

##Sorting Samples into data frames by Sample Type
is_cal = df["Sample Type"] == "Standard"
is_aQC = ~is_cal & df["Sample Name"].str.contains("CCV|ISC|LB|EPA|QC")
is_mQC = ~is_cal & ~is_aQC & df["Sample Name"].str.contains("LCS|LLLCS|LCSD|MB")
is_skip = ~is_cal & ~is_aQC & ~is_mQC & df["Sample Name"].str.contains("DB|blank check"|"AFFF")
is_sam = ~is_cal & ~is_aQC & ~is_mQC & ~is_skip

cal = df[is_cal]
aQC = df[is_aQC]
mQC = df[is_mQC]
sam = df[is_sam]

#This will allow us to label things as target, injection standard, or internal standard, so that we can separate between them.
is_target = (
    df["Component Type"].isin(["Quantifiers", "Qualifiers"]) &
    (df["Component Group Name"] != "Inj Stds")
)

is_inj_std = (
    (df["Component Group Name"] == "Inj Stds") &
    (meth == "100uL")
)

is_int_std = (
    df["Component Type"].isin(["Internal Standard", "Internal Standards"]) &
    (df["Component Group Name"] != "Inj Stds")
)

#Data frames that separate between targets and injection standards

cal_targets = df[is_cal & is_target]
aQC_targets = df[is_aQC & is_target]
mQC_targets = df[is_mQC & is_target]
sam_targets = df[is_sam & is_target]

# injection standards only needed for 100uL method
cal_inj  = df[is_cal & is_inj_std]
aQC_inj  = df[is_aQC & is_inj_std]
mQC_inj  = df[is_mQC & is_inj_std]
sam_inj  = df[is_sam & is_inj_std]

#Creating a matrix of compounds x samples
#Making a make_pivot function to facilitate this
#Reindex is to make sure that the rows are always in compound order

def make_pivot(df_subset, value_col, index_list=None):
    if index_list is None:
        index_list = comnamelist
    return df_subset.pivot_table(
        index="Component Name",
        columns="Sample Name",
        values=value_col,
        aggfunc="first"
    ).reindex(index_list)

#pd.tonumeric converts to numbers, because the cells are read in as strings (text) originally.

sam_PA      = make_pivot(sam_targets, "Area").apply(pd.to_numeric, errors="coerce") #Sample Peak Area
sam_conc    = make_pivot(sam_targets, "Calculated Concentration").apply(pd.to_numeric, errors="coerce") #Sample Calculated concentration
sam_ISPA    = make_pivot(sam_targets, "IS Area").apply(pd.to_numeric, errors="coerce") #Sample IS Peak Area
sam_ISact   = make_pivot(sam_targets, "IS Actual Concentration").apply(pd.to_numeric, errors="coerce") #Sample IS Actual Concentration
sam_rnum    = make_pivot(sam_targets, "Sample Index").apply(pd.to_numeric, errors="coerce") #Sample index
sam_inj_PA  = make_pivot(sam_inj, "Area", index_list=InjSnamelist).apply(pd.to_numeric, errors="coerce") #Injection Standard Peak Area

aQC_PA      = make_pivot(aQC_targets, "Area").apply(pd.to_numeric, errors="coerce") 
aQC_conc    = make_pivot(aQC_targets, "Calculated Concentration").apply(pd.to_numeric, errors="coerce")
aQC_ISPA    = make_pivot(aQC_targets, "IS Area").apply(pd.to_numeric, errors="coerce") 
aQC_ISact   = make_pivot(aQC_targets, "IS Actual Concentration").apply(pd.to_numeric, errors="coerce") 
aQC_rnum    = make_pivot(aQC_targets, "Sample Index").apply(pd.to_numeric, errors="coerce") 
aQC_inj_PA  = make_pivot(aQC_inj, "Area", index_list=InjSnamelist).apply(pd.to_numeric, errors="coerce") 

mQC_PA      = make_pivot(mQC_targets, "Area").apply(pd.to_numeric, errors="coerce") 
mQC_conc    = make_pivot(mQC_targets, "Calculated Concentration").apply(pd.to_numeric, errors="coerce")
mQC_ISPA    = make_pivot(mQC_targets, "IS Area").apply(pd.to_numeric, errors="coerce") 
mQC_ISact   = make_pivot(mQC_targets, "IS Actual Concentration").apply(pd.to_numeric, errors="coerce") 
mQC_rnum    = make_pivot(mQC_targets, "Sample Index").apply(pd.to_numeric, errors="coerce") 
mQC_inj_PA  = make_pivot(mQC_inj, "Area", index_list=InjSnamelist).apply(pd.to_numeric, errors="coerce")

cal_PA      = make_pivot(cal_targets, "Area").apply(pd.to_numeric, errors="coerce") 
cal_conc    = make_pivot(cal_targets, "Calculated Concentration").apply(pd.to_numeric, errors="coerce")
cal_actconc = make_pivot(cal_targets, "Actual Concentration").apply(pd.to_numeric, errors="coerce")
cal_ISPA    = make_pivot(cal_targets, "IS Area").apply(pd.to_numeric, errors="coerce") 
cal_ISact   = make_pivot(cal_targets, "IS Actual Concentration").apply(pd.to_numeric, errors="coerce") 
cal_rnum    = make_pivot(cal_targets, "Sample Index").apply(pd.to_numeric, errors="coerce") 
cal_used    = make_pivot(cal_targets, "Used")
cal_inj_PA  = make_pivot(cal_inj, "Area", index_list=InjSnamelist).apply(pd.to_numeric, errors="coerce")


#MB subtraction from matlab script isn't actually handled in this section despite there being something written. I am not including that stub here.
#MB subtraction is handled in a later section.

In [ ]:
### SECTION 7 ###

##Determining the calibration range

#Sorting cal columns by actual concentration. This just uses the first compound since the cal curve sort is the same for all compounds.

sort_order   = cal_actconc.loc[comnamelist[0]].apply(pd.to_numeric, errors="coerce").argsort()
cal_actconc_s = cal_actconc.iloc[:, sort_order]
cal_conc_s    = cal_conc.iloc[:, sort_order]
cal_used_s    = cal_used.iloc[:, sort_order]

lowlim = {}
highlim = {}

for compound in comnamelist:
    valid = []
    for col in cal_actconc_s.columns:
        actconc = pd.to_numeric(cal_actconc_s.loc[compound, col], errors="coerce")
        conc    = pd.to_numeric(cal_conc_s.loc[compound, col], errors="coerce")
        used    = pd.to_numeric(cal_used_s.loc[compound, col].strip().upper())
        acc     = conc / actconc if actconc !=0 else np.nan
        if 0.7 < acc < 1.3 and used == "TRUE":
            valid.append(actconc)
    
    lowlim[compound]    = valid[0] if valid else np.nan
    highlim[compound]   = valid[-1] if valid else np.nan